# FFA Results QA
A quick look at the results of the Flood Frequency Analysis (LP3 fit to NWIS annual peaks).

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.ticker as mticker
import seaborn as sns
import cartopy.crs as ccrs
import cartopy.feature as cfeature

# DATA_DIR     = Path("../../data/ffa/")
# NWIS_DIR     = Path("../../data/metadata/")

DATA_DIR     = Path("/home/ryan/data/flood_hazard/ffa")
NWIS_DIR     = Path("/home/ryan/data/flood_hazard/metadata")

plt.rcParams['figure.dpi'] = 150
sns.set_theme(style='whitegrid', palette='muted')

CONUS_EXTENT = [-125, -66, 24, 50]

def make_conus_ax(fig, pos=111, title=''):
    subplot_args = pos if isinstance(pos, tuple) else (pos,)
    ax = fig.add_subplot(*subplot_args, projection=ccrs.AlbersEqualArea(
        central_longitude=-96, central_latitude=37.5))
    ax.set_extent(CONUS_EXTENT, crs=ccrs.PlateCarree())
    ax.add_feature(cfeature.LAND,      facecolor='#f5f5f0', zorder=0)
    ax.add_feature(cfeature.OCEAN,     facecolor='#c8e0f0', zorder=0)
    ax.add_feature(cfeature.LAKES,     facecolor='#c8e0f0', zorder=1, alpha=0.6)
    ax.add_feature(cfeature.STATES,    linewidth=0.4,       zorder=2, edgecolor='gray')
    ax.add_feature(cfeature.COASTLINE, linewidth=0.6,       zorder=3)
    ax.add_feature(cfeature.BORDERS,   linewidth=0.6,       zorder=3)
    if title:
        ax.set_title(title)
    return ax

HUC2_NAMES = {
    '01': 'New England',         '02': 'Mid Atlantic',          '03': 'South Atlantic-Gulf',
    '04': 'Great Lakes',         '05': 'Ohio',                  '06': 'Tennessee',
    '07': 'Upper Mississippi',   '08': 'Lower Mississippi',     '09': 'Souris-Red-Rainy',
    '10': 'Missouri',            '11': 'Arkansas-White-Red',    '12': 'Texas-Gulf',
    '13': 'Rio Grande',          '14': 'Upper Colorado',        '15': 'Lower Colorado',
    '16': 'Great Basin',         '17': 'Pacific Northwest',     '18': 'California',
}

## 1. Load Data

In [ ]:
ffa       = pd.read_parquet(DATA_DIR / "flood_frequency.parquet")
peaks     = pd.read_parquet(DATA_DIR / "annual_peaks.parquet")
site_info = pd.read_parquet(NWIS_DIR / "site_info.parquet")
stages    = pd.read_parquet(NWIS_DIR / "flood_stages.parquet")

# Replace Inf return periods with NaN so log-scale plots don't blow up
rp_cols = ['action_return_period_yr', 'flood_return_period_yr',
           'moderate_return_period_yr', 'major_return_period_yr']
ffa[rp_cols] = ffa[rp_cols].replace(np.inf, np.nan)

ffa_ok = ffa[ffa['record_ok'] == True].copy()

print(f"Total sites:          {len(ffa):,}")
print(f"record_ok == True:    {len(ffa_ok):,}")
print(f"Annual peak rows:     {len(peaks):,}")

## 2. Record Quality

In [ ]:
# fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# # n_peaks distribution
# ax = axes[0]
# ax.hist(ffa['n_peaks'], bins=40, edgecolor='white', linewidth=0.3)
# ax.axvline(10, color='red', linestyle='--', linewidth=1.2, label='min=10 (record_ok threshold)')
# ax.axvline(ffa['n_peaks'].median(), color='orange', linestyle='--', linewidth=1.2,
#            label=f"median={ffa['n_peaks'].median():.0f}")
# ax.set_xlabel('Number of annual peaks')
# ax.set_ylabel('Count')
# ax.set_title('Record length distribution')
# ax.legend(fontsize=8)

# # Coverage: how many sites have each threshold
# ax = axes[1]
# aep_cols = ['action_aep', 'flood_aep', 'moderate_aep', 'major_aep']
# labels   = ['Action', 'Flood', 'Moderate', 'Major']
# counts   = [ffa_ok[c].notna().sum() for c in aep_cols]
# ax.bar(labels, counts, edgecolor='white')
# ax.set_ylabel('Sites with valid AEP')
# ax.set_title(f'Threshold coverage  (record_ok, n={len(ffa_ok):,})')
# for i, v in enumerate(counts):
#     ax.text(i, v + 10, str(v), ha='center', fontsize=9)

# plt.tight_layout()
# plt.show()

## 3. LP3 Parameter Distributions

In [ ]:
# lp3_params = ['lp3_skew', 'lp3_loc', 'lp3_scale']
# lp3_labels = ['Skew (γ)', 'Location (log10 μ)', 'Scale (log10 σ)']

# fig, axes = plt.subplots(1, 3, figsize=(14, 3.5))
# for ax, col, label in zip(axes, lp3_params, lp3_labels):
#     data = ffa_ok[col].dropna()
#     ax.hist(data, bins=50, edgecolor='white', linewidth=0.3)
#     ax.axvline(data.median(), color='red', linestyle='--', linewidth=1.2,
#                label=f'median={data.median():.3g}')
#     ax.set_title(label)
#     ax.set_xlabel('Value')
#     ax.legend(fontsize=8)

# fig.suptitle('LP3 fit parameters  (record_ok sites)', fontsize=12)
# plt.tight_layout()
# plt.show()

## 4. Return Period Distributions

In [ ]:
# fig, axes = plt.subplots(1, 4, figsize=(16, 3.5), sharex=True, dpi=150)
# for ax, col, label in zip(axes, rp_cols, ['Action', 'Flood', 'Moderate', 'Major']):
#     data = ffa_ok[col].dropna()
#     log_bins = np.logspace(np.log10(max(data.min(), 0.5)), np.log10(data.quantile(0.99)), 50)
#     ax.hist(data.clip(upper=data.quantile(0.99)), bins=log_bins, edgecolor='white', linewidth=0.3)
#     ax.axvline(data.median(), color='red', linestyle='--', linewidth=1.2,
#                label=f'median={data.median():.1f} yr')
#     ax.set_xscale('log')
#     ax.set_title(f'{label}')
#     ax.set_xlabel('Return period (yr)')
#     ax.legend(fontsize=8)
    
#     ax.set_xlim(1)

# axes[0].set_ylabel('Count')
# fig.suptitle('Return period by flood threshold  (record_ok)', fontsize=12)
# plt.tight_layout()
# plt.show()

## 5. AEP Distributions

In [ ]:
# aep_cols   = ['action_aep', 'flood_aep', 'moderate_aep', 'major_aep']
# aep_labels = ['Action', 'Flood', 'Moderate', 'Major']

# fig, axes = plt.subplots(1, 4, figsize=(16, 3.5), sharex=True)
# for ax, col, label in zip(axes, aep_cols, aep_labels):
#     data = ffa_ok[col].dropna()
#     ax.hist(data, bins=50, edgecolor='white', linewidth=0.3)
#     ax.axvline(data.median(), color='red', linestyle='--', linewidth=1.2,
#                label=f'median={data.median():.3f}')
#     ax.set_title(label)
#     ax.set_xlabel('Annual exceedance probability')
#     ax.legend(fontsize=8)

# axes[0].set_ylabel('Count')
# fig.suptitle('AEP by flood threshold  (record_ok sites)', fontsize=12)
# plt.tight_layout()
# plt.show()

## 6. Maps of Return Periods

In [ ]:
map_df = ffa_ok.merge(
    site_info[['site_no', 'latitude', 'longitude', 'station_name']],
    on='site_no', how='left'
).dropna(subset=['latitude', 'longitude'])

fig = plt.figure(figsize=(18, 12), dpi=150)
for i, (col, label) in enumerate(zip(rp_cols, ['Action', 'Flood', 'Moderate', 'Major']), 1):
    sub  = map_df.dropna(subset=[col])
    # norm = mcolors.LogNorm(vmin=sub[col].quantile(0.02), vmax=sub[col].quantile(0.98))
    norm = mcolors.LogNorm(vmin=1, vmax=6e3)
    ax   = make_conus_ax(fig, pos=(2, 2, i), title=f'{label} return period (yr)')
    sc   = ax.scatter(
        sub['longitude'], sub['latitude'],
        c=sub[col], cmap='plasma_r', norm=norm,
        s=12, alpha=0.8, linewidths=0,
        transform=ccrs.PlateCarree(), zorder=4
    )
    plt.colorbar(sc, ax=ax, orientation='horizontal', shrink=0.8, pad=0.04,
                 label='Return period (yr)')

fig.suptitle(f'LP3 return periods by flood threshold  |  n={len(map_df):,}', fontsize=13)
plt.tight_layout()
plt.show()

## 7. Return Periods by HUC2

In [ ]:
huc_df = ffa_ok.merge(site_info[['site_no', 'huc8']], on='site_no', how='left')
huc_df['huc2'] = huc_df['huc8'].astype('Int64').astype(str).str.zfill(8).str[:2]
huc_df['huc2_name'] = huc_df['huc2'].map(HUC2_NAMES)
huc_df = huc_df[huc_df['huc2'].isin(HUC2_NAMES)]

print(f"n={len(huc_df):,} gages across {huc_df['huc2'].nunique()} HUC2s")

In [ ]:
# # Sort HUC2s by median action return period
# order_huc2 = (
#     huc_df.dropna(subset=['action_return_period_yr', 'huc2_name'])
#     .groupby('huc2_name')['action_return_period_yr']
#     .median()
#     .sort_values()
#     .index.tolist()
# )

# counts = huc_df['huc2_name'].value_counts()
# order_labeled = [f'{h}  (n={counts.get(h, 0)})' for h in order_huc2]
# name_to_label = dict(zip(order_huc2, order_labeled))
# huc_df['huc2_label'] = huc_df['huc2_name'].map(name_to_label)

# fig, axes = plt.subplots(1, 2, figsize=(16, 7), dpi=150, sharey=True)

# for ax, col, label in zip(
#     axes,
#     ['action_return_period_yr', 'flood_return_period_yr'],
#     ['Action', 'Flood'],
# ):
#     sub = huc_df.dropna(subset=[col, 'huc2_label'])
#     sub = sub[sub[col] < sub[col].quantile(0.99)]  # clip outliers for readability
#     sns.boxplot(
#         data=sub, y='huc2_label', x=col,
#         order=order_labeled, ax=ax,
#         color='steelblue', linewidth=0.8, fliersize=2, width=0.6,
#     )
#     ax.set_title(f'{label} return period by HUC2')
#     ax.set_xlabel('Return period (yr)')
#     ax.set_ylabel('')
#     ax.axvline(sub[col].median(), color='red', linestyle='--', linewidth=0.9,
#                label=f'national median={sub[col].median():.1f} yr')
#     ax.legend(fontsize=8)
    
#     ax.set_xlim(0,100)

# plt.tight_layout()
# plt.show()

## 8. Major / Moderate Thresholds by HUC2

In [ ]:
# fig, axes = plt.subplots(1, 2, figsize=(16, 7), dpi=150, sharey=True)

# for ax, col, label in zip(
#     axes,
#     ['moderate_return_period_yr', 'major_return_period_yr'],
#     ['Moderate', 'Major'],
# ):
#     sub = huc_df.dropna(subset=[col, 'huc2_label'])
#     sub = sub[sub[col] < sub[col].quantile(0.99)]
#     sns.boxplot(
#         data=sub, y='huc2_label', x=col,
#         order=order_labeled, ax=ax,
#         color='coral', linewidth=0.8, fliersize=2, width=0.6,
#     )
#     ax.set_title(f'{label} return period by HUC2')
#     ax.set_xlabel('Return period (yr)')
#     ax.set_ylabel('')
#     ax.axvline(sub[col].median(), color='red', linestyle='--', linewidth=0.9,
#                label=f'national median={sub[col].median():.1f} yr')
#     ax.legend(fontsize=8)
    
#     ax.set_xlim(0,1000)

# plt.tight_layout()
# plt.show()

## 9. Return Period vs. Drainage Area

In [ ]:
# scatter_df = ffa_ok.merge(
#     site_info[['site_no', 'drainage_area_sqmi']],
#     on='site_no', how='left'
# ).dropna(subset=['drainage_area_sqmi'])

# fig, axes = plt.subplots(1, 4, figsize=(16, 4), sharex=True, sharey=True)
# for ax, col, label in zip(axes, rp_cols, ['Action', 'Flood', 'Moderate', 'Major']):
#     sub = scatter_df.dropna(subset=[col])
#     sub = sub[(sub['drainage_area_sqmi'] > 0) & (sub[col] > 0)]
#     ax.scatter(sub['drainage_area_sqmi'], sub[col],
#                s=6, alpha=0.4, linewidths=0)
#     ax.set_xscale('log')
#     ax.set_yscale('log')
#     ax.set_title(label)
#     ax.set_xlabel('Drainage area (mi²)')
    
#     ax.set_ylim(1, 1e3)

# axes[0].set_ylabel('Return period (yr)')
# fig.suptitle('Return period vs. drainage area', fontsize=12)
# plt.tight_layout()
# plt.show()

## 10. Annual Peak Series — Example Sites

In [ ]:
import matplotlib.lines as mlines
from scipy.stats import pearson3

# One site per HUC2 — pick the site with the most annual peaks
sample_sites = (
    ffa_ok
    .merge(huc_df[['site_no', 'huc2']], on='site_no', how='left')
    .dropna(subset=['huc2', 'lp3_skew'])
    .sort_values('n_peaks', ascending=False)
    .drop_duplicates('huc2')
    .sort_values('huc2')
    .head(18)['site_no']
    .tolist()
)

fig, axes = plt.subplots(4, 5, figsize=(16, 10), sharex=True)
axes = axes.ravel()

for ax, site in zip(axes, sample_sites):
    site_peaks = peaks[(peaks['site_no'] == site) & (peaks['peak_va'] > 0)]['peak_va'].dropna()
    row = ffa_ok[ffa_ok['site_no'] == site].iloc[0]

    if site_peaks.empty or pd.isna(row['lp3_skew']):
        ax.set_visible(False)
        continue

    # Empirical plotting positions (Weibull)
    sorted_peaks = np.sort(site_peaks)[::-1]
    n = len(sorted_peaks)
    emp_rp = (n + 1) / np.arange(1, n + 1)

    # LP3 curve
    aep_curve = np.logspace(-3, 0, 200)
    rp_curve  = 1 / aep_curve
    flow_curve = 10 ** pearson3.ppf(1 - aep_curve,
                                    row['lp3_skew'], row['lp3_loc'], row['lp3_scale'])

    ax.scatter(emp_rp, sorted_peaks, s=20, marker='x', color='dimgray', linewidth=0.5, zorder=0)
    ax.plot(rp_curve, flow_curve, 'k', linewidth=1)
    ax.scatter(ffa[rp_cols[0]][ffa['site_no'] == site], stages['action_flow_cfs'][stages['site_no'] == site],
               zorder=5, color='#FFFF00')
    ax.scatter(ffa[rp_cols[1]][ffa['site_no'] == site], stages['flood_flow_cfs'][stages['site_no'] == site],
               zorder=5, color='#FF9900')
    ax.scatter(ffa[rp_cols[2]][ffa['site_no'] == site], stages['moderate_flow_cfs'][stages['site_no'] == site],
               zorder=5, color='#FF0000')
    ax.scatter(ffa[rp_cols[3]][ffa['site_no'] == site], stages['major_flow_cfs'][stages['site_no'] == site],
               zorder=5, color="#CC33FF")

    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.set_title(f'{site} (n={n})', fontsize=9)
    ax.set_xlim(1e0, 1e4)

# Hide all unused axes, then reclaim the first one for the legend
n_sites = len(sample_sites)
for ax in axes[n_sites:]:
    ax.set_visible(False)

legend_elements = [
    mlines.Line2D([0], [0], color='dimgray', marker='x', linestyle='None', markersize=6, linewidth=0.5, label='Obs. peaks'),
    mlines.Line2D([0], [0], color='k', linewidth=1, label='LP3 fit'),
    mlines.Line2D([0], [0], color='#FFFF00', marker='o', linestyle='None', markersize=7, label='Action'),
    mlines.Line2D([0], [0], color='#FF9900', marker='o', linestyle='None', markersize=7, label='Minor'),
    mlines.Line2D([0], [0], color='#FF0000', marker='o', linestyle='None', markersize=7, label='Moderate'),
    mlines.Line2D([0], [0], color='#CC33FF', marker='o', linestyle='None', markersize=7, label='Major'),
]
legend_ax = axes[n_sites]
legend_ax.set_visible(True)
legend_ax.axis('off')
legend_ax.legend(handles=legend_elements, loc='center', fontsize=9, frameon=False)

fig.supxlabel('Return period (yr)')
fig.supylabel('Peak flow (cfs)')
fig.suptitle('LP3 frequency curves — sample sites', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# Sample site locations

fig = plt.figure(figsize=(6.5, 4), dpi=150)

sub  = map_df[map_df['site_no'].isin(sample_sites)]

ax   = make_conus_ax(fig, pos=(1, 1, 1), title='Sample Sites')
sc   = ax.scatter(
    sub['longitude'], sub['latitude'],
    c='red', marker='*', s=100, alpha=1, linewidths=0,
    transform=ccrs.PlateCarree(), zorder=4
)

# fig.suptitle(f'Sample Sites', fontsize=13)
plt.tight_layout()
plt.show()